# Clase 8 — Funciones y reutilización para IA

En la Clase 7 limpiamos registros y trabajamos con datasets usando listas y diccionarios.  
El problema es que, si copiamos y pegamos la misma lógica una y otra vez, el código se vuelve difícil de mantener.

Para eso existen las **funciones**: bloques de código reutilizables que reciben datos, hacen una tarea concreta y devuelven un resultado.

**Objetivos de la clase:**
- Escribir funciones con `def`, parámetros y `return`.
- Reutilizar la limpieza y validación del dataset.
- Encapsular el clasificador manual y KNN en funciones.
- Armar un mini pipeline reproducible para evaluar varias frutas.


---
## 1. ¿Qué resuelve una función?

Una función sirve cuando querés:
- evitar repetir código,
- darle un nombre claro a una tarea,
- probar una pieza por separado,
- reutilizar la misma lógica con datos distintos.

Estructura general:
```python
def nombre(parametros):
    # hacer algo
    return resultado
```


In [ ]:
# Una función muy simple
def distancia_peso(a, b):
    return abs(a - b)

print("Distancia entre 175g y 182g:", distancia_peso(175, 182))

def promedio_pesos(pesos):
    total = 0
    for peso in pesos:
        total += peso
    return total / len(pesos)

print("Promedio de pesos:", promedio_pesos([182, 143, 170, 130]))


---
## 2. Normalizar y validar como pasos reutilizables

En la clase anterior limpiamos registros con bucles.  
Ahora vamos a encapsular esa lógica en funciones con nombres claros.

Separar responsabilidades ayuda mucho:
- `normalizar_registro(...)` corrige formato,
- `registro_valido(...)` decide si las features mínimas están presentes.

Nota: para **predecir** una fruta nueva, no siempre conocemos la clase real.  
Por eso la validación básica se enfoca primero en las **features** (`peso`, `color`, `dulzura`).


In [ ]:
def normalizar_registro(registro):
    peso = registro.get("peso")
    color = registro.get("color", "")
    dulzura = registro.get("dulzura")
    clase = registro.get("clase", "")

    if peso != None and peso != "":
        peso = int(peso)
    else:
        peso = None

    if dulzura != None and dulzura != "":
        dulzura = int(dulzura)
    else:
        dulzura = None

    return {
        "peso": peso,
        "color": color,
        "dulzura": dulzura,
        "clase": clase,
    }

crudo = {"peso": "182", "color": "rojo", "dulzura": "7", "clase": "manzana"}
limpio = normalizar_registro(crudo)


def registro_valido(registro):
    return (
        registro["peso"] != None
        and registro["color"] != ""
        and registro["dulzura"] != None
    )

print("Registro limpio:", limpio)
print("¿Registro válido?", registro_valido(limpio))


In [ ]:
def clasificar_fruta(peso, color, dulzura):
    if peso > 150 and color in ["rojo", "verde"] and dulzura >= 6:
        return "manzana"
    if peso <= 160 and color == "naranja" and dulzura >= 5:
        return "naranja"
    if color == "amarillo" and dulzura < 4:
        return "limon"
    if peso < 20:
        return "uva"
    return "desconocida"


casos = [
    (182, "rojo", 1),
    (143, "naranja", 6),
    (95, "amarillo", 3),
    (12, "morado", 8),
]

for peso, color, dulzura in casos:
    print((peso, color, dulzura), "->", clasificar_fruta(peso, color, dulzura))


---
## 3. Encapsular KNN manual en una función

En la Clase 6 implementamos KNN usando bucles.  
Ahora la idea es que esa lógica quede disponible como una pieza reutilizable.

Nuestra versión inicial de KNN va a usar solo `peso` para medir distancia.  
No es un modelo completo, pero alcanza para entender la mecánica.


In [ ]:
# ┌──────────────────────────────────────────┐
# │  predecir_knn(dataset, nueva_fruta, k=1) │
# └──────────────────────────────────────────┘
#
#   para cada fruta del dataset:
#         │
#         ▼
#   ¿es válida y tiene clase? ──No──► se descarta
#         │
#        Sí
#         │
#         ▼
#   distancia = distancia_peso(nueva_fruta, fruta)
#         │
#         ▼
#   distancias.append((distancia, clase))
#
#   distancias.sort()          (la más cercana queda primera)
#         │
#         ▼
#   vecinos = distancias[:k]   (los k más cercanos)
#         │
#         ▼
#   contar cuántas veces aparece cada clase entre los vecinos
#         │
#         ▼
#   devolver la clase más frecuente

def predecir_knn(dataset, nueva_fruta, k=1):
    distancias = []

    for fruta in dataset:
        if registro_valido(fruta) and fruta.get("clase", "") != "":
            distancia = distancia_peso(nueva_fruta["peso"], fruta["peso"])
            distancias.append((distancia, fruta["clase"]))

    distancias.sort()
    vecinos = distancias[:k]

    if len(vecinos) == 0:
        return "sin_datos"

    conteo = {}
    for distancia, clase in vecinos:
        if clase in conteo:
            conteo[clase] += 1
        else:
            conteo[clase] = 1

    clase_mas_frecuente = None
    mayor_conteo = 0

    for clase in conteo:
        if conteo[clase] > mayor_conteo:
            mayor_conteo = conteo[clase]
            clase_mas_frecuente = clase

    return clase_mas_frecuente


dataset_entrenamiento = [
    {"peso": 182, "color": "rojo",     "dulzura": 7, "clase": "manzana"},
    {"peso": 143, "color": "naranja",  "dulzura": 6, "clase": "naranja"},
    {"peso": 170, "color": "verde",    "dulzura": 6, "clase": "manzana"},
    {"peso": 130, "color": "naranja",  "dulzura": 5, "clase": "naranja"},
]

nueva_fruta = {"peso": 155, "color": "naranja", "dulzura": 6, "clase": ""}

print("KNN k=1:", predecir_knn(dataset_entrenamiento, nueva_fruta, k=1))
print("KNN k=2:", predecir_knn(dataset_entrenamiento, nueva_fruta, k=2))


---
## 4. Encadenar funciones: mini pipeline reproducible

Cuando cada paso está encapsulado, podemos construir un flujo completo:
- limpiar,
- validar,
- predecir,
- comparar,
- resumir resultados.

Eso ya se parece mucho más a un script de trabajo real.


In [ ]:
# ┌───────────────────────────────────┐
# │   evaluar_modelo(dataset)          │
# └───────────────────────────────────┘
#
# PASO 1: limpiar y validar cada registro
#
#   dataset (crudo)
#         │
#         ▼
#   normalizar_registro(registro)
#         │
#         ▼
#   ¿registro_valido? ──No──► invalidos += 1
#         │
#        Sí
#         │
#         ▼
#   dataset_limpio.append(registro)
#
# PASO 2: evaluar cada fruta de dataset_limpio
#
#   para cada fruta:
#         │
#         ├──► clasificar_fruta(...) ──► pred_arbol
#         │         │
#         │         ▼
#         │    ¿pred_arbol == clase real?
#         │      Sí ──► arbol_correctas += 1
#         │      No ──► errores.append(...)
#         │
#         └──► predecir_knn(resto del dataset, fruta) ──► pred_knn
#                   │
#                   ▼
#              ¿pred_knn == clase real?
#                Sí ──► knn_correctas += 1
#                No ──► errores.append(...)
#
# PASO 3: devolver el reporte con todos los contadores

def evaluar_modelo(dataset):
    dataset_limpio = []
    reporte = {
        "evaluados": 0,
        "invalidos": 0,
        "arbol_correctas": 0,
        "knn_correctas": 0,
        "errores": [],
    }

    for registro in dataset:
        limpio = normalizar_registro(registro)
        if not registro_valido(limpio) or limpio["clase"] == "":
            reporte["invalidos"] += 1
        else:
            dataset_limpio.append(limpio)

    for i, fruta in enumerate(dataset_limpio):
        reporte["evaluados"] += 1

        pred_arbol = clasificar_fruta(
            fruta["peso"],
            fruta["color"],
            fruta["dulzura"],
        )

        dataset_sin_fruta_actual = []
        for j, otra_fruta in enumerate(dataset_limpio):
            if j != i:
                dataset_sin_fruta_actual.append(otra_fruta)

        pred_knn = predecir_knn(dataset_sin_fruta_actual, fruta, k=1)

        if pred_arbol == fruta["clase"]:
            reporte["arbol_correctas"] += 1
        else:
            reporte["errores"].append(
                {"modelo": "arbol", "esperada": fruta["clase"], "predicha": pred_arbol}
            )

        if pred_knn == fruta["clase"]:
            reporte["knn_correctas"] += 1
        else:
            reporte["errores"].append(
                {"modelo": "knn", "esperada": fruta["clase"], "predicha": pred_knn}
            )

    return reporte


dataset_evaluacion = [
    {"peso": "182", "color": "rojo",     "dulzura": "7", "clase": "manzana"},
    {"peso": "143", "color": "naranja",  "dulzura": "6", "clase": "naranja"},
    {"peso": "170", "color": "verde",    "dulzura": "6", "clase": "manzana"},
    {"peso": "130", "color": "naranja",  "dulzura": "5", "clase": "naranja"},
    {"peso": "",    "color": "amarillo", "dulzura": "3", "clase": "limon"},
]

print(evaluar_modelo(dataset_evaluacion))


---
## 5. Un script reproducible es una secuencia clara de funciones

La ventaja real no es solo "que el código quede lindo".  
La ventaja es que ahora podés aplicar la misma lógica sobre:
- otro dataset,
- más frutas,
- nuevos casos de prueba,
- o una evaluación completa sin copiar bloques enteros.


In [ ]:
dataset_demo = [
    {"peso": 182, "color": "rojo",     "dulzura": 7, "clase": "manzana"},
    {"peso": 143, "color": "naranja",  "dulzura": 6, "clase": "naranja"},
    {"peso": 170, "color": "verde",    "dulzura": 6, "clase": "manzana"},
    {"peso": 12,  "color": "morado",   "dulzura": 8, "clase": "uva"},
]

nuevas_frutas = [
    {"peso": "175", "color": "verde",   "dulzura": "6"},
    {"peso": "138", "color": "naranja", "dulzura": "5"},
    {"peso": "9",   "color": "morado",  "dulzura": "8"},
]

print("Reporte del dataset:", evaluar_modelo(dataset_demo))
print()
print("Predicciones para frutas nuevas:")

for fruta in nuevas_frutas:
    limpia = normalizar_registro(fruta)
    if registro_valido(limpia):
        pred_arbol = clasificar_fruta(limpia["peso"], limpia["color"], limpia["dulzura"])
        pred_knn = predecir_knn(dataset_demo, limpia, k=1)
        print(limpia, "-> árbol:", pred_arbol, "| knn:", pred_knn)
    else:
        print(limpia, "-> registro inválido")


---
## 📝 Actividad 1 — Refactorizar un bloque repetido en una función

**Consigna:**
Creá una función `mostrar_resumen(registro)` que devuelva un texto corto con el formato:

```python
"manzana roja de 182g (dulzura 7)"
```


In [ ]:
# ACTIVIDAD 1: refactorizar un bloque repetido en una función
registros = [
    {"peso": 182, "color": "rojo",     "dulzura": 7, "clase": "manzana"},
    {"peso": 143, "color": "naranja",  "dulzura": 6, "clase": "naranja"},
    {"peso": 12,  "color": "morado",   "dulzura": 8, "clase": "uva"},
]

def mostrar_resumen(registro):
    # Devuelve un texto corto con clase, color, peso y dulzura.
    texto = "pendiente"
    # TODO: armar el texto usando las claves del registro
    return texto


for registro in registros:
    print(mostrar_resumen(registro))


---
### Actividad 2 — Evaluar varias frutas sin duplicar lógica

**Consigna:**
- Normalizá cada fruta nueva.
- Validala.
- Si es válida, generá dos predicciones:
  - una con `clasificar_fruta(...)`,
  - otra con `predecir_knn(...)`.
- Guardá todo en la lista `predicciones`.


In [ ]:
# ACTIVIDAD 2: evaluar varias frutas sin duplicar lógica
dataset = [
    {"peso": 182, "color": "rojo",     "dulzura": 7, "clase": "manzana"},
    {"peso": 143, "color": "naranja",  "dulzura": 6, "clase": "naranja"},
    {"peso": 170, "color": "verde",    "dulzura": 6, "clase": "manzana"},
    {"peso": 130, "color": "naranja",  "dulzura": 5, "clase": "naranja"},
]

nuevas_frutas = [
    {"peso": "178", "color": "verde",    "dulzura": "6"},
    {"peso": "136", "color": "naranja",  "dulzura": "5"},
    {"peso": "",    "color": "amarillo", "dulzura": "3"},
]

predicciones = []

for fruta in nuevas_frutas:
    fruta_limpia = normalizar_registro(fruta)

    if registro_valido(fruta_limpia):
        pred_arbol = "pendiente"
        pred_knn = "pendiente"
    else:
        pred_arbol = "registro invalido"
        pred_knn = "registro invalido"

    predicciones.append(
        {
            "entrada": fruta_limpia,
            "arbol": pred_arbol,
            "knn": pred_knn,
        }
    )

print(predicciones)


---
### Actividad 3 — Construir un mini pipeline con reporte final

**Consigna:**
Tomá `dataset_mixto`, aplicá las funciones del notebook y completá el reporte:
- cantidad de evaluados,
- cantidad de inválidos,
- aciertos del árbol,
- aciertos de KNN,
- errores detectados.


In [ ]:
# ACTIVIDAD 3: construir un mini pipeline con reporte final
dataset_mixto = [
    {"peso": "182", "color": "rojo",     "dulzura": "7", "clase": "manzana"},
    {"peso": "143", "color": "naranja",  "dulzura": "6", "clase": "naranja"},
    {"peso": "170", "color": "verde",    "dulzura": "6", "clase": "manzana"},
    {"peso": "",    "color": "morado",   "dulzura": "8", "clase": "uva"},
    {"peso": "95",  "color": "amarillo", "dulzura": "3", "clase": "limon"},
]

reporte = {
    "evaluados": 0,
    "invalidos": 0,
    "arbol_correctas": 0,
    "knn_correctas": 0,
    "errores": [],
}

# TODO:
# 1. normalizar y validar cada registro
# 2. usar clasificar_fruta(...) y predecir_knn(...)
# 3. completar el reporte final

print(reporte)


---
## ✅ Resumen

| Función | Rol en el flujo |
|---|---|
| `normalizar_registro(...)` | Limpia formato |
| `registro_valido(...)` | Verifica features mínimas |
| `clasificar_fruta(...)` | Regla manual tipo árbol |
| `distancia_peso(...)` | Distancia básica para KNN |
| `predecir_knn(...)` | Predicción por vecinos |
| `evaluar_modelo(...)` | Resumen de desempeño |

**Lo que construiste hoy:**
- funciones pequeñas con una responsabilidad clara,
- un clasificador reutilizable,
- un KNN manual encapsulado,
- y un pipeline sencillo para evaluar datos.

**Lo que viene:**
- **Clase 9**: vamos a conocer NumPy, la librería que usa casi todo proyecto de IA para trabajar con muchos números a la vez, sin escribir bucles para cada operación.
